In [ ]:
# 1. 환경설정
import sys, subprocess, importlib.util, os, json, shutil, re
from pathlib import Path

packages = {
    "catboost": "catboost", "lightgbm": "lightgbm", "xgboost": "xgboost",
    "shap": "shap", "openpyxl": "openpyxl",
    "koreanize_matplotlib": "koreanize-matplotlib",
}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import shap
from IPython import get_ipython
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, auc, confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, DMatrix
from catboost import CatBoostClassifier, Pool
from google.colab import files

FILE_PATH = Path("NEDIS_응급표본자료_2020_2024.xlsx")
OUT = Path("NEDIS_results")
OUT.mkdir(exist_ok=True)
SEED = 2024
THREADS = min(4, os.cpu_count() or 1)
TEAL, ORANGE = "#16728B", "#CE8144"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False, "figure.facecolor": "white",
    "savefig.facecolor": "white", "font.size": 9,
})

def save_figure(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def save_table(frame, name):
    frame.to_csv(OUT / f"{name}.csv", encoding="utf-8-sig")
    display(frame)

# 실행 중인 셀 원문을 보관해 ZIP 안에 재실행 가능한 코드와 노트북을 넣는다.
ip = get_ipython()
cell_source = ip.history_manager.input_hist_raw[-1]

# 2. 데이터 로드 및 필터링
if not FILE_PATH.exists():
    print("원본 NEDIS 엑셀 또는 CSV 파일 한 개를 선택하세요.")
    uploaded = files.upload()
    candidates = [Path(n) for n in uploaded if Path(n).suffix.lower() in {".xlsx", ".csv"}]
    if len(candidates) != 1:
        raise ValueError("원본 .xlsx 또는 .csv 파일 한 개를 선택해 주세요.")
    FILE_PATH = candidates[0]

if FILE_PATH.suffix.lower() == ".csv":
    try:
        raw = pd.read_csv(FILE_PATH, dtype=str, encoding="utf-8-sig")
    except UnicodeDecodeError:
        raw = pd.read_csv(FILE_PATH, dtype=str, encoding="cp949")
else:
    with pd.ExcelFile(FILE_PATH) as book:
        sheet = "데이터셋" if "데이터셋" in book.sheet_names else book.sheet_names[0]
        raw = book.parse(sheet, dtype=str)

schema = [
    ("PTMIEMAR", "응급의료기관 지역", "Facility region"),
    ("PTMIEMCL", "응급의료기관 종별", "Center tier"),
    ("PTMIINDT", "내원일자", "Arrival date"),
    ("PTMIINTM", "내원시간", "Arrival time"),
    ("PTMIBRTD", "연령", "Age group"),
    ("PTMISEXX", "성별", "Sex"),
    ("PTMIPTAR", "환자주소지역", "Residence region"),
    ("PTMIAKDT", "발병일자", "Onset date"),
    ("PTMIAKTM", "발병시간", "Onset time"),
    ("PTMIDGKD", "질병여부", "Disease status"),
    ("PTMIARCF", "의도성 여부", "Intent"),
    ("PTMIARCS", "손상기전", "Injury mechanism"),
    ("PTMIINRT", "내원경로", "Arrival route"),
    ("PTMIINMN", "내원수단", "Transport mode"),
    ("PTMIMNSY", "주증상-1", "Chief complaint (UMLS)"),
    ("PTMIMSSR", "주증상-1 일련번호", "Chief complaint serial"),
    ("PTMIEMSY", "응급증상 해당 여부", "Emergency symptom status"),
]
def key(v):
    return re.sub(r"\s+", "", str(v)).casefold()

aliases = {key(a): en for code, ko, en in schema for a in [code, ko, en]}
aliases[key("PTMIERMAR")] = "Facility region"
aliases[key("응급기관 종별")] = "Center tier"
raw = raw.rename(columns=lambda c: aliases.get(key(c), c))
fields = [en for _, _, en in schema]
absent = set(fields) - set(raw.columns)
if absent:
    raise ValueError(f"원본 데이터 시트에서 찾지 못한 열: {sorted(absent)}")
df = raw[fields].copy().apply(lambda s: s.astype("string").str.strip().replace("", pd.NA))

def decode(series, mapping):
    lookup = {key(k): v for k, v in mapping.items()}
    def one(v):
        if pd.isna(v):
            return "Unknown"
        token = key(v)
        m = re.fullmatch(r"(\d+)(?:\.0+)?(?::.*)?", token)
        if m:
            token = str(int(m[1]))
        return lookup.get(token, str(v))
    return series.map(one)

def labels(codes, korean, english):
    return {k: e for c, h, e in zip(codes, korean, english) for k in [str(c), h, e]}

transport = labels(
    range(1, 6),
    ["119구급차", "의료기관 구급차", "기타 구급차", "경찰차 등 공공차량", "항공이송"],
    ["119 ambulance", "Hospital ambulance", "Other ambulance", "Public vehicle", "Air transport"],
)
df["Facility region"] = decode(df["Facility region"], {"8": "Gyeonggi", "경기도": "Gyeonggi"})
df["Center tier"] = decode(df["Center tier"], {
    "1": "Regional", "권역응급의료센터": "Regional",
    "2": "Local", "지역응급의료센터": "Local",
})
df["Transport mode"] = decode(df["Transport mode"], transport)
region_ok = df["Facility region"].eq("Gyeonggi")
tier_ok = region_ok & df["Center tier"].isin(["Regional", "Local"])
keep = tier_ok & df["Transport mode"].isin(transport.values())
save_table(pd.Series(
    [len(df), region_ok.sum(), tier_ok.sum(), keep.sum()],
    index=["Input", "Gyeonggi", "Eligible tier", "Eligible transport"],
    name="Records").to_frame(), "cohort_flow")
df = df.loc[keep].reset_index(drop=True)
y = df["Center tier"].eq("Regional").astype(int)

# 성별은 첨부 코드북에 따라 1=남성, 2=여성
df["Sex"] = decode(df["Sex"], labels([1, 2], ["남성", "여성"], ["Male", "Female"]))
save_table(df["Sex"].value_counts().rename("Records").to_frame(), "sex_counts")

# 3. 파생변수 두 개와 모델 입력
def parse_day(s):
    text = s.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
    text = text.mask(text.isin(["1111", "9999", "11111111", "99999999"]))
    text = text.str.replace(r" 00:00:00$", "", regex=True).str.replace(r"[-/.]", "", regex=True)
    return pd.to_datetime(text.where(text.str.fullmatch(r"\d{8}", na=False)),
                          format="%Y%m%d", errors="coerce")

def parse_minutes(s, onset=False):
    text = s.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
    if onset:
        text = text.mask(text.isin(["1111", "9999"]))
    colon = text.str.extract(r"^(\d{1,2}):(\d{2})(?::00)?$")
    compact = text.where(text.str.fullmatch(r"\d{1,4}", na=False)).str.zfill(4)
    hour = pd.to_numeric(colon[0].fillna(compact.str[:2]), errors="coerce")
    minute = pd.to_numeric(colon[1].fillna(compact.str[2:]), errors="coerce")
    return (hour * 60 + minute).where(hour.between(0, 23) & minute.between(0, 59)).astype(float)

ad, od = parse_day(df["Arrival date"]), parse_day(df["Onset date"])
am, om = parse_minutes(df["Arrival time"]), parse_minutes(df["Onset time"], onset=True)
arrival_dt = ad + pd.to_timedelta(am, unit="m")
onset_dt = od + pd.to_timedelta(om, unit="m")
elapsed = (arrival_dt - onset_dt).dt.total_seconds() / 60
DUR, PERIOD = "Onset-to-arrival interval (min)", "Onset period"
UMLS, SERIAL = "Chief complaint (UMLS)", "Chief complaint serial"
period_order = ["Overnight (00–05)", "Morning (06–11)", "Afternoon (12–17)", "Evening (18–23)"]
df[DUR] = elapsed.where(elapsed >= 0)
df[PERIOD] = pd.cut(om, [0, 360, 720, 1080, 1440], right=False,
                    labels=period_order).astype("string").fillna("Unknown")
df["Arrival date"] = (ad - pd.Timestamp("1970-01-01")).dt.days
df["Onset date"] = (od - pd.Timestamp("1970-01-01")).dt.days
df["Arrival time"], df["Onset time"] = am, om
age = pd.to_numeric(df["Age group"], errors="coerce")
df["Age group"] = age.where(age.between(1, 17))

df["Intent"] = decode(df["Intent"], labels(
    [1, 2, 3, 8, 9], ["비의도적 사고", "자해/자살", "폭력/타살", "기타", "미상"],
    ["Unintentional", "Self-harm", "Assault", "Other", "Unknown"]))
df["Injury mechanism"] = decode(df["Injury mechanism"], labels(
    [1, 3, 4, 8, 9, 10, 11, 20, 30, 40, 50, 60, 70, 80, 88, 99],
    ["자동차교통사고", "자전거교통사고", "오토바이교통사고", "기타교통사고", "미상교통사고",
     "추락", "미끄러짐", "부딪힘", "베임/찔림", "기계", "화상", "익수", "중독", "질식/목맴",
     "기타(성폭력,감전 등)", "미상"],
    ["Car crash", "Bicycle crash", "Motorcycle crash", "Other traffic", "Unspecified traffic",
     "Fall from height", "Slip or trip", "Struck by object", "Cut or stab", "Machinery", "Burn",
     "Drowning", "Poisoning", "Asphyxia", "Other injury", "Unknown"]))

num = ["Arrival date", "Arrival time", "Age group", "Onset date", "Onset time", DUR]
X = df.drop(columns=["Facility region", "Center tier"]).copy()
cat = [c for c in X if c not in num]
for c in num:
    X[c] = X[c].astype(float)
for c in cat:
    X[c] = X[c].replace("미상", pd.NA).fillna("UNKNOWN" if c in [UMLS, SERIAL] else "Unknown").astype(str)
assert X.shape[1] == 17
save_table(pd.Series({
    "Missing onset date": od.isna().sum(), "Missing onset time": om.isna().sum(),
    "Negative intervals removed": elapsed.lt(0).sum(),
    "Unavailable intervals": df[DUR].isna().sum(),
}).rename("Records").to_frame(), "time_quality")

# 4. EDA 및 600 DPI 그림
tier_order = ["Local", "Regional"]
counts = df["Center tier"].value_counts().reindex(tier_order, fill_value=0)
cross = pd.crosstab(df["Transport mode"], df["Center tier"]).reindex(columns=tier_order, fill_value=0)
save_table(pd.DataFrame({"Records": counts, "Percent": 100*counts/len(df)}), "class_distribution")
save_table(cross, "transport_by_tier")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), layout="constrained")
counts.plot.bar(ax=axes[0], color=[TEAL, ORANGE], rot=0)
cross.plot.barh(stacked=True, ax=axes[1], color=[TEAL, ORANGE])
axes[0].set(xlabel="Center tier", ylabel="Encounters")
axes[1].set(xlabel="Encounters", ylabel="")
save_figure(fig, "01_cohort")

stats = pd.concat([df[DUR].describe().rename("All").to_frame().T,
                   df.groupby("Center tier")[DUR].describe()])
save_table(stats, "interval_statistics")
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
for pos, (tier, color) in enumerate(zip(tier_order, [TEAL, ORANGE]), 1):
    values = df.loc[df["Center tier"].eq(tier), DUR].dropna()
    if len(values):
        ax.boxplot(values, positions=[pos], vert=False, patch_artist=True,
                   boxprops={"facecolor": color}, medianprops={"color": "black"},
                   flierprops={"marker": ".", "markersize": 2, "alpha": .3})
ax.set_xscale("symlog", linthresh=10)
ax.set(yticks=[1, 2], yticklabels=tier_order, xlabel="Onset-to-arrival interval (min)")
save_figure(fig, "02_interval")

for i, c in enumerate([PERIOD, "Injury mechanism", "Intent"], 3):
    rates = y.groupby(X[c]).agg(["sum", "count"])
    rates.columns = ["Regional", "Total"]
    rates["Percent regional"] = 100*rates.Regional/rates.Total
    save_table(rates, f"rate_{i}")
    shown = (rates.reindex([s for s in period_order+["Unknown"] if s in rates.index])
             if c == PERIOD else rates.nlargest(10, "Total"))
    fig, ax = plt.subplots(figsize=(7, max(3, .36*len(shown)+1)), layout="constrained")
    ax.barh(shown.index, shown["Percent regional"], color=TEAL)
    ax.invert_yaxis()
    ax.set(xlim=(0, 100), xlabel="Regional center share (%)", title=c)
    save_figure(fig, f"0{i}_regional_share")

corr = X[num].corr(method="spearman")
save_table(corr, "spearman_correlations")
fig, ax = plt.subplots(figsize=(6.5, 5), layout="constrained")
palette = LinearSegmentedColormap.from_list("corr", [ORANGE, "white", TEAL])
im = ax.imshow(corr, cmap=palette, vmin=-1, vmax=1)
names = ["Arrival date", "Arrival time", "Age group", "Onset date", "Onset time", "Interval"]
ax.set_xticks(range(6), names, rotation=40, ha="right")
ax.set_yticks(range(6), names)
for i, j in np.ndindex(6, 6):
    v = corr.iloc[i, j]
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v)>.8 else "black")
fig.colorbar(im, ax=ax, label="Spearman correlation")
save_figure(fig, "06_correlation")

# 5. 층화 8:2 분할, 학습 세트 5-fold, 다섯 모델 × 세 가지 주증상 조건
train, test = train_test_split(np.arange(len(y)), test_size=.2, stratify=y, random_state=SEED)
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y.iloc[train]))
models = ["Logistic regression", "Random forest", "LightGBM", "XGBoost", "CatBoost"]
variants = {"UMLS only": [SERIAL], "Serial only": [UMLS], "Both": []}

def make_model(name, columns, target):
    cats = [c for c in columns if c in cat]
    if name == "CatBoost":
        return CatBoostClassifier(iterations=120, depth=5, learning_rate=.1, verbose=False,
            cat_features=cats, auto_class_weights="Balanced", random_seed=SEED,
            thread_count=THREADS, allow_writing_files=False)
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                              StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True,
                              dtype=np.float32), cats),
    ], sparse_threshold=1)
    if name == "Logistic regression":
        est = LogisticRegression(class_weight="balanced", solver="liblinear",
                                 max_iter=500, random_state=SEED)
    elif name == "Random forest":
        est = RandomForestClassifier(n_estimators=80, max_depth=16, min_samples_leaf=3,
            class_weight="balanced", random_state=SEED, n_jobs=THREADS)
    elif name == "LightGBM":
        est = LGBMClassifier(n_estimators=120, learning_rate=.08, verbosity=-1,
            class_weight="balanced", random_state=SEED, n_jobs=THREADS)
    else:
        est = XGBClassifier(n_estimators=120, max_depth=5, learning_rate=.1,
            tree_method="hist", eval_metric="logloss", random_state=SEED, n_jobs=THREADS,
            scale_pos_weight=(target==0).sum()/(target==1).sum())
    return make_pipeline(pre, est)

def scores(truth, probability):
    label = probability >= .5
    p, r, _ = precision_recall_curve(truth, probability)
    return {
        "Accuracy": accuracy_score(truth, label),
        "Precision": precision_score(truth, label, zero_division=0),
        "Recall": recall_score(truth, label, zero_division=0),
        "F1": f1_score(truth, label, zero_division=0),
        "ROC-AUC": roc_auc_score(truth, probability), "PR-AUC": auc(r, p),
    }

records = []
for variant, omit in variants.items():
    data = X.drop(columns=omit)
    for name in models:
        print(f"학습 중 | {variant} | {name}", flush=True)
        for k, (a, b) in enumerate(folds, 1):
            tr, va = train[a], train[b]
            model = make_model(name, data.columns, y.iloc[tr])
            model.fit(data.iloc[tr], y.iloc[tr])
            p = model.predict_proba(data.iloc[va])[:, 1]
            records.append(dict(Variant=variant, Model=name, Fold=k, **scores(y.iloc[va], p)))
cv = pd.DataFrame(records)
save_table(cv, "cv_each_fold")
metric_names = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
grouped = cv.groupby(["Variant", "Model"])[metric_names]
save_table(grouped.agg(["mean", "std"]), "cv_summary")
means = grouped.mean()
save_table(means[["ROC-AUC", "F1", "Recall"]].unstack("Variant"), "ablation")
rank = means.loc["Both"].sort_values(["ROC-AUC", "Recall", "F1"], ascending=False)
winner = rank.index[0]
xai_name = next(name for name in rank.index if name in ["CatBoost", "XGBoost"])

held = []
fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
for ax, name in zip(axes.flat, models):
    model = make_model(name, X.columns, y.iloc[train])
    model.fit(X.iloc[train], y.iloc[train])
    p = model.predict_proba(X.iloc[test])[:, 1]
    held.append(dict(Model=name, **scores(y.iloc[test], p)))
    if name == xai_name:
        xai_model = model
    cm = confusion_matrix(y.iloc[test], p>=.5, labels=[0, 1])
    share = cm/cm.sum(axis=1, keepdims=True)
    ax.imshow(share, cmap=LinearSegmentedColormap.from_list("cm", ["white", TEAL]), vmin=0, vmax=1)
    for i, j in np.ndindex(2, 2):
        ax.text(j, i, f"{cm[i,j]:,}\n{share[i,j]:.1%}", ha="center", va="center",
                color="white" if share[i,j]>.65 else "black")
    ax.set(xticks=[0,1], yticks=[0,1], xticklabels=tier_order, yticklabels=tier_order,
           title=name, xlabel="Predicted tier", ylabel="Recorded tier")
axes.flat[-1].set_visible(False)
save_figure(fig, "07_confusion_matrices")
results = pd.DataFrame(held).set_index("Model")
save_table(results, "test_metrics")

# 6. SHAP 설명 | 두 후보 중 학습 세트 교차검증 ROC-AUC가 높은 모델
sample = X.iloc[test].sample(min(800, len(test)), random_state=SEED)
if xai_name == "CatBoost":
    pool = Pool(sample, cat_features=cat)
    contrib = xai_model.get_feature_importance(pool, type="ShapValues", thread_count=THREADS)
    phi, base = contrib[:, :-1], contrib[:, -1]
    margin = xai_model.predict(pool, prediction_type="RawFormulaVal")
else:
    pre = xai_model.steps[0][1]
    booster = xai_model.steps[-1][1].get_booster()
    mat = DMatrix(pre.transform(sample))
    contrib = booster.predict(mat, pred_contribs=True)
    base = contrib[:, -1]
    widths = [1]*len(num) + [len(level) for level in pre.named_transformers_["cat"].categories_]
    grouped_phi = np.add.reduceat(contrib[:, :-1], np.cumsum([0]+widths[:-1]), axis=1)
    phi = pd.DataFrame(grouped_phi, columns=num+cat)[X.columns].to_numpy()
    margin = booster.predict(mat, output_margin=True)
np.testing.assert_allclose(phi.sum(axis=1)+base, margin, atol=1e-4, rtol=1e-4)

# 범주형 변수에 임의 순서를 부여하지 않도록 beeswarm의 색은 수치형에만 적용
color_data = sample[num].reindex(columns=X.columns).to_numpy(dtype=float)
ex = shap.Explanation(values=phi, base_values=base, data=color_data,
                      feature_names=X.columns.tolist())
np.random.seed(SEED)
shap.plots.beeswarm(ex, max_display=len(X.columns), show=False, plot_size=(8, 6.5),
                    color=LinearSegmentedColormap.from_list("shap_value", [TEAL, ORANGE]))
plt.gca().set_xlabel("SHAP value (log-odds for regional center)")
save_figure(plt.gcf(), "08_shap_beeswarm")

importance = pd.Series(np.abs(phi).mean(axis=0), index=X.columns, name="Mean absolute SHAP")
save_table(importance.sort_values(ascending=False).to_frame(), "shap_importance")
fig, ax = plt.subplots(figsize=(8, 5.5), layout="constrained")
importance.sort_values().plot.barh(ax=ax, color=TEAL)
ax.set_xlabel("Mean absolute SHAP value (log-odds)")
save_figure(fig, "09_shap_importance")

rng = np.random.default_rng(SEED)
for i, c in enumerate([DUR, PERIOD, UMLS, "Injury mechanism", "Age group"], 10):
    values = sample[c]
    effects = phi[:, X.columns.get_loc(c)]
    fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
    if c in num:
        ax.scatter(values, effects, color=TEAL, s=10, alpha=.5, linewidths=0)
        if c == DUR:
            ax.set_xscale("symlog", linthresh=10)
    else:
        levels = ([s for s in period_order+["Unknown"] if s in set(values)]
                  if c==PERIOD else values.value_counts().head(10).index.tolist())
        shown = values.where(values.isin(levels), "Remaining levels")
        if "Remaining levels" in set(shown):
            levels.append("Remaining levels")
        positions = shown.map({s: j for j,s in enumerate(levels)}).to_numpy(dtype=float)
        ax.scatter(positions+rng.uniform(-.2,.2,len(shown)), effects,
                   color=TEAL, s=10, alpha=.5, linewidths=0)
        ax.set_xticks(range(len(levels)), levels, rotation=35, ha="right")
    ax.axhline(0, color="gray", lw=.7)
    ax.set(xlabel=c, ylabel="SHAP value (log-odds)")
    save_figure(fig, f"{i}_dependence")

# 7. 한국어 요약, 재실행용 노트북, ZIP 다운로드
report = f"""# 경기도 응급 이송 환자의 실제 도착 응급의료기관 종별 분류

분석 표본은 {len(X):,}건이며 권역센터 도착은 {y.sum():,}건({y.mean():.1%})이다.
학습 세트 {len(train):,}건과 시험 세트 {len(test):,}건으로 층화 분할했다.
성별은 코드북의 1=남성, 2=여성을 적용했다. 연령은 17개 연령군 코드다.
발병부터 도착까지 경과시간의 중앙값은 {df[DUR].median():.1f}분이다.

두 주증상을 모두 입력한 모델 중 평균 검증 ROC-AUC 1위는 {winner}였다.
평균 검증 ROC-AUC는 {rank.loc[winner,'ROC-AUC']:.3f}이다.
별도 시험 세트 ROC-AUC는 {results.loc[winner,'ROC-AUC']:.3f},
권역센터 Recall은 {results.loc[winner,'Recall']:.3f}, F1은 {results.loc[winner,'F1']:.3f}이다.
Recall은 실제 권역센터 도착 중 모델이 권역센터로 분류한 비율이다.
SHAP은 {xai_name}의 시험 표본 {len(sample):,}건을 사용했다.
평균 절대 SHAP 상위 세 변수는 {', '.join(importance.nlargest(3).index)}이다.

목표변수는 기록된 도착 기관이다. 실제 도착시각과 도착까지 걸린 시간은 목적지 결정 후에 확정된다.
이 결과는 후향적 분류 성능을 나타낸다. 현장 최적 배정, 이동거리 절감, 생존 개선을 평가하려면
목적지 결정 시점의 정보와 독립적인 적정 이송 기준 및 결과 검증이 필요하다.
"""
(OUT/"report_ko.md").write_text(report, encoding="utf-8")
print(report)

if "NEDIS_응급표본자료_2020_2024.xlsx" in cell_source:
    (OUT/"NEDIS_Gyeonggi_ED.py").write_text(cell_source, encoding="utf-8")
    nb = {
        "nbformat": 4, "nbformat_minor": 5,
        "metadata": {"kernelspec": {"name":"python3","display_name":"Python 3","language":"python"}},
        "cells": [{"cell_type":"code","id":"nedis-study","metadata":{},"execution_count":None,
                   "outputs":[],"source":cell_source.splitlines(keepends=True)}],
    }
    (OUT/"NEDIS_Gyeonggi_ED.ipynb").write_text(
        json.dumps(nb, ensure_ascii=False, indent=1), encoding="utf-8")
else:
    print("현재 셀 원문을 가져오지 못했습니다. 코드 사본은 Colab의 파일 → 다운로드에서 받으세요.")

archive = shutil.make_archive("NEDIS_all", "zip", root_dir=OUT)
print("생성 완료:", archive)
files.download(archive)